# 01 — Public Sample Data Overview

This notebook introduces the public sample datasets used by the Prediction Market Execution Lab.

The datasets in `data/sample/` are anonymized, downsampled, normalized, and intended for demonstration only. They are sufficient to inspect schema, reproduce the public reports, and understand the analysis workflow. They should not be interpreted as the full private dataset or as complete empirical performance evidence.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("..") / "data" / "sample"
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

## Load public sample files

The public sample includes candidate signals, execution attempts, rejection records, market settlements, and tick-level quote snapshots.

In [ ]:
files = {
    "candidates": DATA_DIR / "candidates_sample.csv",
    "executions": DATA_DIR / "executions_sample.csv",
    "rejections": DATA_DIR / "rejections_sample.csv",
    "settlements": DATA_DIR / "settlements_sample.csv",
    "tick_snapshots": DATA_DIR / "tick_snapshots_sample.csv",
}

datasets = {name: pd.read_csv(path) for name, path in files.items()}

inventory = pd.DataFrame(
    {
        "dataset": name,
        "path": str(path),
        "rows": len(df),
        "columns": df.shape[1],
    }
    for name, path in files.items()
    for df in [datasets[name]]
)
inventory

## Schema summary

The schema is intentionally field-filtered. Identifiers are anonymized, monetary values are bucketed or normalized, and private operational fields are excluded.

In [ ]:
schema_rows = []
for name, df in datasets.items():
    for column in df.columns:
        schema_rows.append(
            {
                "dataset": name,
                "column": column,
                "dtype": str(df[column].dtype),
                "non_null": int(df[column].notna().sum()),
                "null_share": float(df[column].isna().mean()),
            }
        )

schema = pd.DataFrame(schema_rows)
schema.head(40)

## Time coverage

The public sample keeps enough timestamp structure to demonstrate the workflow, but it is not a complete historical export.

In [ ]:
time_columns = {
    "candidates": "recorded_at",
    "executions": "recorded_at",
    "rejections": "recorded_at",
    "settlements_start": "market_start_utc",
    "settlements_end": "market_end_utc",
    "tick_snapshots": "timestamp",
}

time_summary = []
for label, column in time_columns.items():
    dataset_name = label.split("_")[0] if label.startswith("settlements") else label
    df = datasets[dataset_name].copy()
    values = pd.to_datetime(df[column], utc=True, errors="coerce")
    time_summary.append(
        {
            "series": label,
            "column": column,
            "non_null": int(values.notna().sum()),
            "min_utc": values.min(),
            "max_utc": values.max(),
        }
    )

pd.DataFrame(time_summary)

## Join-key coverage

The primary safe join key is the anonymized `market_id`. Candidate and execution rows also contain anonymized `candidate_id`, but not all rows are expected to join one-to-one because the public sample is downsampled.

In [ ]:
market_sets = {
    name: set(df["market_id"].dropna().astype(str))
    for name, df in datasets.items()
    if "market_id" in df.columns
}

join_rows = []
for left, left_ids in market_sets.items():
    for right, right_ids in market_sets.items():
        if left >= right:
            continue
        join_rows.append(
            {
                "left": left,
                "right": right,
                "left_markets": len(left_ids),
                "right_markets": len(right_ids),
                "intersection": len(left_ids & right_ids),
            }
        )

pd.DataFrame(join_rows)

## Tick-level quote fields

Tick snapshots preserve quote, implied probability, and fair probability fields needed for replay-style diagnostics.

In [ ]:
tick = datasets["tick_snapshots"].copy()
quote_columns = [
    "timestamp",
    "market_id",
    "yes_bid",
    "yes_ask",
    "yes_mid",
    "yes_spread",
    "pm_implied_up",
    "fair_yes",
    "tau_seconds",
    "quote_complete",
]
existing_quote_columns = [column for column in quote_columns if column in tick.columns]
tick[existing_quote_columns].head(10)

## Settlement fields

Settlement rows use normalized and bucketed values. This supports demonstration-level PnL attribution without exposing private order-level ledger details.

In [ ]:
settlements = datasets["settlements"].copy()
settlement_columns = [
    "market_id",
    "market_start_utc",
    "market_end_utc",
    "resolved_side",
    "total_cost_bucket",
    "gross_payout_bucket",
    "net_pnl_normalized",
    "mode_observed",
]
settlements[[column for column in settlement_columns if column in settlements.columns]].head(10)

## Public-safe ML diagnostics fields

The public candidate and execution samples include scalar ML and fill-probability diagnostics exported from the private ledger. They intentionally exclude model paths, feature-name lists, raw feature JSON, raw responses, order IDs, and wallet-related fields.


In [ ]:
ml_columns = [
    "ml_filter_enabled",
    "ml_predicted_ev",
    "ml_min_ev",
    "ml_passed",
    "ml_reason",
    "fill_probability",
    "fill_prob_min_probability",
    "fill_prob_passed",
    "fill_prob_reason",
]
coverage_rows = []
for name in ["candidates", "executions"]:
    frame = datasets[name]
    for column in ml_columns:
        if column in frame.columns:
            coverage_rows.append(
                {
                    "dataset": name,
                    "column": column,
                    "non_null_rows": int(frame[column].notna().sum()),
                    "rows": len(frame),
                }
            )
pd.DataFrame(coverage_rows)


## Takeaway

The public sample is designed to make the research pipeline reproducible: inspect sample schema, evaluate execution-quality diagnostics, generate sample reports, and build public-facing figures. It is not designed to reconstruct private market activity or support live execution.